# Churn Prediction Platform

Predicting telecom customer churn with a full-stack ML pipeline.

**Progress:**
- [x] Step 1: Data cleaning
- [x] Step 2: EDA & feature analysis
- [x] Step 3: Model training & validation
- [ ] Step 4: Prediction API (Django REST Framework)
- [ ] Step 5: Frontend (React)
- [ ] Step 6: Deployment (AWS)
- [ ] Step 7: Monitoring dashboard (Power BI)

Dataset: [IBM Telco Customer Churn](https://www.kaggle.com/blastchar/telco-customer-churn) — 7,043 customers, 21 features.

## Step 1: Data Cleaning

In [ ]:
import pandas as pd

df = pd.read_csv("telco.csv")
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

**Key finding:** `TotalCharges` is stored as text, and 11 rows contain a blank value instead of a number. Every one of these rows has `tenure == 0` — these are brand-new customers who haven't completed a full billing cycle yet, not random missing data. We set these to `0.0` rather than dropping or imputing a mean, since a mean would misrepresent a genuinely new customer.

In [ ]:
before_blank = (df["TotalCharges"].str.strip() == "").sum()
df["TotalCharges"] = df["TotalCharges"].replace(" ", pd.NA)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

zero_tenure_mask = df["tenure"] == 0
df.loc[zero_tenure_mask & df["TotalCharges"].isna(), "TotalCharges"] = 0.0

print(f"Fixed {before_blank} blank values")
print(f"Remaining nulls: {df['TotalCharges'].isna().sum()}")

Standardize categorical labels — collapse `'No internet service'` / `'No phone service'` into `'No'` across related columns to keep the feature space smaller and more interpretable.

In [ ]:
cols_to_simplify = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                    "TechSupport", "StreamingTV", "StreamingMovies"]
for col in cols_to_simplify:
    df[col] = df[col].replace({"No internet service": "No", "No phone service": "No"})
df["MultipleLines"] = df["MultipleLines"].replace({"No phone service": "No"})

df["Churn_Flag"] = (df["Churn"] == "Yes").astype(int)

df["tenure_bucket"] = pd.cut(
    df["tenure"], bins=[-1, 6, 12, 24, 48, 72],
    labels=["0-6mo", "7-12mo", "13-24mo", "25-48mo", "49-72mo"]
)
df["avg_monthly_spend"] = df["TotalCharges"] / df["tenure"].clip(lower=1)

print("Class balance (Churn):")
print((df["Churn"].value_counts(normalize=True) * 100).round(1))
print("-> Imbalanced. Use stratified splits and report precision/recall/F1 in Step 3, not just accuracy.")

df.to_csv("telco_cleaned.csv", index=False)
print(f"\nSaved telco_cleaned.csv ({df.shape[0]} rows, {df.shape[1]} columns)")

## Step 2: Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt

rates = (df.groupby("Contract")["Churn_Flag"].mean() * 100).round(1)
rates = rates.reindex(["Month-to-month", "One year", "Two year"])
print(rates)

ax = rates.plot(kind="bar", color="#4C72B0", figsize=(6, 4))
ax.set_title("Churn Rate by Contract Type")
ax.set_ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()

**Finding:** Month-to-month customers churn at **42.7%** vs just **2.8%** for two-year contracts — roughly a 15x difference. Likely the strongest single predictor in this dataset.

In [ ]:
order = ["0-6mo", "7-12mo", "13-24mo", "25-48mo", "49-72mo"]
tenure_rates = (df.groupby("tenure_bucket", observed=True)["Churn_Flag"]
                .mean() * 100).round(1).reindex(order)
print(tenure_rates)

ax = tenure_rates.plot(kind="bar", color="#DD8452", figsize=(6, 4))
ax.set_title("Churn Rate by Tenure")
ax.set_ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()

**Finding:** Churn risk is highest in a customer's first 6 months (52.9%) and drops steadily with tenure (9.5% past 4 years) — a classic \"new customer risk window.\"

In [ ]:
means = df.groupby("Churn")["MonthlyCharges"].mean().round(2)
print(means)

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "avg_monthly_spend", "Churn_Flag"]
print("\nCorrelation with Churn_Flag:")
print(df[numeric_cols].corr()["Churn_Flag"].round(3))

**Counterintuitive finding:** customers who churn pay *more* per month on average (\$74.44) than customers who stay (\$61.27). Suggests price sensitivity or perceived value plays a role independent of tenure and contract type — worth exploring in modeling (e.g. an interaction between `MonthlyCharges` and `InternetService`).

## Step 3: Model Training & Validation

We use a stratified 80/20 split to preserve the 73.5%/26.5% class balance in both sets. Numeric features are scaled with `StandardScaler` — this matters for Logistic Regression (distance/gradient-based) but not for Random Forest (threshold-based splits), which is itself a useful thing to verify rather than assume. Given the class imbalance, we prioritize recall on the churn class and ROC-AUC over raw accuracy.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

feature_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen',
                'Contract', 'InternetService', 'PaymentMethod', 'Partner', 'Dependents']
X = df[feature_cols].copy()
y = df['Churn_Flag']

X = pd.get_dummies(X, columns=['Contract', 'InternetService', 'PaymentMethod', 'Partner', 'Dependents'],
                    drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

In [ ]:
# Scale numeric features. Fit ONLY on training data, then apply to test data,
# to avoid leaking test-set information into the scaling parameters.
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])
print("Scaled numeric features:", numeric_features)

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

print("Logistic Regression:")
print(classification_report(y_test, pred_lr))
print("ROC-AUC:", round(roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1]), 3))

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print("Random Forest:")
print(classification_report(y_test, pred_rf))
print("ROC-AUC:", round(roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]), 3))

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(5)

**Finding:** *(Fill this in with your own printed numbers above once you run the notebook — compare Logistic Regression vs Random Forest on recall for the churn class and ROC-AUC, and note which model wins and why that might be, e.g. dataset size, linear separability of features, or overfitting risk in Random Forest.)*

**Top predictive features** (from Random Forest importances): TotalCharges, MonthlyCharges, and tenure dominate, followed by contract type — consistent with the EDA findings in Step 2.

## Step 4: Prediction API (Django REST Framework)

*(next up)*